In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

import joblib
import random
import numpy as np
import torch
from matplotlib import pyplot as plt
from syd import make_viewer, Viewer
from tqdm import tqdm

from vrAnalysis.database import get_database
from vrAnalysis.helpers import Timer, edge2center
from vrAnalysis.helpers.plotting import beeswarm, format_spines, add_scaled_limits
from vrAnalysis.processors.placefields import get_placefield, get_frame_behavior
from dimilibi import gaussian_filter, fit_powerlaw_decay, fit_powerlaw_derivatives
from dimensionality_manuscript.configs import get_data_config
from dimensionality_manuscript import StimSpaceSpectraConfig, ResultsStore, ResultsAggregator
from dimensionality_manuscript.registry import PopulationRegistry
from dimensionality_manuscript.regression_models.hyperparameters import PlaceFieldHyperparameters

plt.rcParams["font.size"] = 12

# get session database
sessiondb = get_database("vrSessions")

# get population registry
registry = PopulationRegistry(registry_params=get_data_config("even").to_registry_params())

In [2]:
sessions = sessiondb.iter_sessions(imaging=True)
store = ResultsStore()
cfg = StimSpaceSpectraConfig()
results = ResultsAggregator(cfg, store, sessions)

In [3]:
results.param_axes

{'activity_parameters_name': ['raw', 'default'],
 'smooth_widths': [(None, None), (5.0, 5.0), (5.0, None)],
 'reliability_fraction_active_thresholds': [(None, None), (0.3, 0.1)],
 'include_iti': [False, True]}

In [6]:
def xvals(x):
    return np.arange(x.shape[1]) + 1

def tuple_float_map(tf: tuple | None, to_value: bool = True):
    def _map_float_or_none(x: str) -> float | None:
        if x == "None":
            return None
        return float(x)
    if to_value:
        value_split = tf.split("-")
        return tuple(map(_map_float_or_none, value_split))
    else:
        return f"{tf[0]}-{tf[1]}"

class SpectraViewer(Viewer):
    def __init__(self, results: ResultsAggregator):
        self.results = results
        for param, values in results.param_axes.items():
            if param == "smooth_widths":
                # We need to convert this weird tuple to a mapping
                smooth_widths_as_strings = [tuple_float_map(sw, to_value=False) for sw in values]
                self.add_selection(param, options=smooth_widths_as_strings)
            elif param == "reliability_fraction_active_thresholds":
                rel_fa_as_strings = [tuple_float_map(rf, to_value=False) for rf in values]
                self.add_selection(param, options=rel_fa_as_strings)
            else:
                self.add_selection(param, options=values)
        
        self.add_boolean("average_by_mouse", value=True)

        preferred_state = {
            "smooth_widths": tuple_float_map((5.0, None), to_value=False),
            "reliability_fraction_active_thresholds": tuple_float_map((None, None), to_value=False),
            "activity_parameters_name": "default",
        }
        for key, value in preferred_state.items():
            self.update_selection(key, value=value)       

        self.add_float("ylim_min", value=-6.0, min=-12.0, max=0.0)
        self.add_float("ylim_max", value=-0.5, min=-12.0, max=0.0)

    def plot(self, state: dict):
        selection_keys = {key: state[key] for key in self.results.param_axes}
        selection_keys["smooth_widths"] = tuple_float_map(selection_keys["smooth_widths"], to_value=True)
        selection_keys["reliability_fraction_active_thresholds"] = tuple_float_map(selection_keys["reliability_fraction_active_thresholds"], to_value=True)
        result = self.results.sel(**selection_keys, avg_by_mouse=state["average_by_mouse"])

        full_sum = np.nansum(result["ff"], axis=1, keepdims=True)
        ss_cv = result["ss_cv"] / full_sum
        sf_cv = result["sf_cv"] / full_sum
        ss = result["ss_direct"] / full_sum
        sf = result["sf_direct"] / full_sum
        ff = result["ff"] / full_sum

        fig, ax = plt.subplots(1, 3, figsize=(9, 4), layout="constrained", width_ratios=[1, 0.5, 0.5])
    
        ss_color = "blue"
        sf_color = "orange"
        ff_color = "black"
        each_alpha = 0.3
        fig, ax = plt.subplots(1, 2, figsize=(5, 4), layout="constrained", width_ratios=[1, 0.5])

        ax[0].plot(xvals(ss), ss.T, color=ss_color, alpha=each_alpha)
        ax[0].plot(xvals(sf), sf_cv.T, color=sf_color, alpha=each_alpha)
        ax[0].plot(xvals(ff), ff.T, color=ff_color, alpha=each_alpha)
        ax[0].plot(xvals(ss), np.nanmean(ss, axis=0), color=ss_color, label="SS")
        ax[0].plot(xvals(sf), np.nanmean(sf, axis=0), color=sf_color, label="SF")
        ax[0].plot(xvals(ff), np.nanmean(ff, axis=0), color=ff_color, label="FF")
        ax[0].plot(xvals(ss_cv), np.nanmean(ss_cv, axis=0), color=ss_color, linestyle="--", label="SS")
        ax[0].plot(xvals(sf_cv), np.nanmean(sf_cv, axis=0), color=sf_color, linestyle="--", label="SF")
        ax[0].set_xscale("log")
        ax[0].set_yscale("log")
        ylim = (10 **state["ylim_min"], 10 **state["ylim_max"])
        ax[0].set_ylim(*ylim)

        # Also plot ratio of cumulative variance
        sf_total = np.nansum(sf, axis=1)
        sf_cv_total = np.nansum(sf_cv, axis=1)
        xticks = [0, 1]
        xticklabels = ["Direct", "CV"]
        beewidth = 0.2
        ax[1].plot(beewidth * beeswarm(sf_total), 100*sf_total, color="black", linestyle="none", linewidth=0.5, marker="o", markersize=3)
        ax[1].plot(1 + beewidth * beeswarm(sf_cv_total), 100*sf_cv_total, color="black", linestyle="none", linewidth=0.5, marker="o", markersize=3)
        ax[1].set_title(f"{np.round(np.mean(sf_total[0]) * 100, 1)}% || {np.round(np.mean(sf_cv_total[0]) * 100, 1)}%")
        ax[1].set_ylim(-10, 110)
        ax[1].set_xlim(-0.5, max(xticks) + 0.5)
        ax[1].set_xticks(xticks, labels=xticklabels, rotation=45, ha="right")
        ax[1].set_ylabel("Ratio (SVR)")

        return fig

viewer = SpectraViewer(results)
viewer.show()

C:\Users\Andrew\Documents\GitHub\vrAnalysis\dimensionality_manuscript\pipeline\aggregate.py:44: RuntimeWarning: Mean of empty slice
  out[i] = np.nanmean(arr[mouse_names == mouse], axis=0)
